In [12]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'

import numpy as np
import tensorflow as tf
import csv
import os
from PIL import Image

In [29]:
def getImgAndLab(file_path, folder_path):
    
    names = []
    annot = []

    with open(file_path, newline='') as csvfile:
        reader = csv.reader(csvfile)
        for row in reader:
            annot.append(row)

    for entry_name in os.listdir(folder_path):
        full_path = os.path.join(folder_path, entry_name)
        if(not entry_name == "_annotations.csv"):
            names.append(entry_name)

    images = []
    labels = []

    for i in range(len(annot)):
        for j in range(len(names)):
            try:
                index = annot[i].index(names[j])

                path = folder_path + annot[i][0]
                original_image = Image.open(path)
                box = (int(annot[i][4]), int(annot[i][5]), int(annot[i][6]), int(annot[i][7]))
                cropped_image = original_image.crop(box)
                images.append(cropped_image)
                labels.append(annot[i][3])
                #print(cropped_image)
            except ValueError:
                continue

        target_size  = (120, 120)
    processed_imgs = []

    for img in images:
        img_resized = img.resize(target_size)
        img_rgb = img_resized.convert('RGB')
        processed_imgs.append(img_rgb)

    images_np = np.array(processed_imgs, dtype=np.float32) / 255.0

    unique_labels = sorted(list(set(labels)))
    label_to_int = {label: i for i, label in enumerate(unique_labels)}
    labels_as_integers = [label_to_int[label] for label in labels]

    labels_np = np.array(labels_as_integers)

    return images_np, labels_np


In [32]:
train_images, train_labels = getImgAndLab("./data/train/_annotations.csv", "./data/train/")
test_images, test_labels = getImgAndLab("./data/test/_annotations.csv", "./data/test/")


model = tf.keras.models.Sequential([
  tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2, 2),
  tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2,2),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer="RMSprop", loss="sparse_categorical_crossentropy", metrics=['accuracy'])
model.fit(train_images, train_labels, epochs=10)
test_loss, test_accuracy = model.evaluate(test_images, test_labels)
print ('Test loss: {}, Test accuracy: {}'.format(test_loss, test_accuracy))

Epoch 1/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 11s 81ms/step - accuracy: 0.7438 - loss: 0.7317
Epoch 2/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 11s 81ms/step - accuracy: 0.8381 - loss: 0.4545
Epoch 3/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 11s 83ms/step - accuracy: 0.8919 - loss: 0.3162
Epoch 4/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 11s 83ms/step - accuracy: 0.9170 - loss: 0.2414
Epoch 5/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 12s 92ms/step - accuracy: 0.9396 - loss: 0.1788
Epoch 6/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 12s 88ms/step - accuracy: 0.9543 - loss: 0.1374
Epoch 7/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 12s 91ms/step - accuracy: 0.9616 - loss: 0.1172
Epoch 8/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 12s 93ms/step - accuracy: 0.9668 - loss: 0.1000
Epoch 9/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 13s 96ms/step - accuracy: 0.9753 - loss: 0.0854
Epoch 10/10
132/132 ━━━━━━━━━━━━━━━━━━━━ 12s 91ms/step - accuracy: 0.9775 - loss: 0.0623
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9435 - loss: 0.2215
Test loss: 0.22149083018302917, Test accuracy